## Ejercicio 2: Escalamiento de tickets de soporte técnico

### Ficha PEAS

- **Percepción (S):** tiempo_espera_minutos del ticket, nivel_urgencia reportado ("baja", "media" o "alta"), y si cliente_premium (True/False).
- **Acciones (A):** mantener el ticket en el nivel actual, escalar a nivel 2, o escalar a nivel 3 (soporte especializado).
- **Entorno (E):** la mesa de ayuda de la empresa, recibiendo tickets de distintos clientes de forma continua.
- **Objetivo:** resolver los tickets urgentes o de clientes premium con prioridad, sin saturar innecesariamente los niveles superiores de soporte.
- **Medida de desempeño (P):** tiempo promedio de resolución de tickets urgentes, y satisfacción del cliente premium, frente al uso eficiente de los recursos de soporte especializado.

### Justificación de las reglas

La urgencia alta se escala de inmediato porque, por definición, representa un problema crítico que no puede esperar sin importar otras condiciones. Para la urgencia media, se combina con el tiempo de espera porque un ticket medio que lleva esperando mucho tiempo se vuelve, en la práctica, tan urgente como uno alto — no escalarlo perjudicaría al cliente. El estatus premium se usa como un factor adicional en los casos de urgencia baja y media, porque estos clientes tienen una expectativa de servicio más alta contractualmente, así que se les da un margen de espera más corto antes de escalar, sin llegar a tratarlos igual que una urgencia alta real.

### Código

In [3]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    # Urgencia alta siempre escala de inmediato, sin importar tiempo de espera ni tipo de cliente
    if nivel_urgencia == "alta":
        return "escalar a nivel 3", "urgencia alta reportada por el cliente"
    
    elif nivel_urgencia == "media":
        # Escala si ya espero demasiado, o si es cliente premium (aunque el tiempo sea bajo)
        if tiempo_espera_minutos > 30 or cliente_premium:
            motivo = f"urgencia media con tiempo de espera de {tiempo_espera_minutos} min"
            if cliente_premium:
                motivo += " y cliente premium"
            return "escalar a nivel 2", motivo
        else:
            return "mantener en nivel 1", f"urgencia media con tiempo de espera bajo ({tiempo_espera_minutos} min)"
    
    else:  # urgencia baja
        # Solo escala si es premium Y ademas ya espero mucho (las dos condiciones juntas)
        if cliente_premium and tiempo_espera_minutos > 60:
            return "escalar a nivel 2", f"urgencia baja pero cliente premium con espera de {tiempo_espera_minutos} min"
        else:
            return "mantener en nivel 1", f"urgencia baja, sin condiciones que ameriten escalar"

### Simulación y pruebas

In [2]:
casos = [
    (10, "alta", False),      # urgencia alta -> siempre escala, sin importar tiempo
    (45, "media", False),     # media + tiempo alto, sin premium -> escalar nivel 2
    (15, "media", False),     # media + tiempo bajo, sin premium -> mantener
    (20, "media", True),      # media + tiempo bajo, PERO premium -> escalar nivel 2 (compiten)
    (70, "baja", True),       # baja + premium + tiempo alto -> escalar nivel 2
    (70, "baja", False),      # baja + tiempo alto, sin premium -> mantener (no es suficiente)
]

for tiempo, urgencia, premium in casos:
    accion, motivo = agente_soporte(tiempo, urgencia, premium)
    print(f"Tiempo: {tiempo} min, Urgencia: {urgencia}, Premium: {premium} -> {accion} ({motivo})")

Tiempo: 10 min, Urgencia: alta, Premium: False -> escalar a nivel 3 (urgencia alta reportada por el cliente)
Tiempo: 45 min, Urgencia: media, Premium: False -> escalar a nivel 2 (urgencia media con tiempo de espera de 45 min)
Tiempo: 15 min, Urgencia: media, Premium: False -> mantener en nivel 1 (urgencia media con tiempo de espera bajo (15 min))
Tiempo: 20 min, Urgencia: media, Premium: True -> escalar a nivel 2 (urgencia media con tiempo de espera de 20 min y cliente premium)
Tiempo: 70 min, Urgencia: baja, Premium: True -> escalar a nivel 2 (urgencia baja pero cliente premium con espera de 70 min)
Tiempo: 70 min, Urgencia: baja, Premium: False -> mantener en nivel 1 (urgencia baja, sin condiciones que ameriten escalar)


### Reflexión
A diferencia del agente de crédito, aquí las tres condiciones (urgencia, tiempo de espera, cliente premium) pueden competir entre sí en lugar de simplemente sumarse, lo que obligó a pensar el orden de evaluación con más cuidado (revisar primero el caso más crítico, urgencia alta, antes de combinar los demás factores).